In [1]:
from intake_esgf import ESGFCatalog
import matplotlib.pyplot as plt
import xarray as xr
import xesmf as xe

def anomaly_calculation(model, clim):
    ano_list = []
    for i in range(1,13):
        ano_list.append(model.sel(time=model.time[model.time.dt.month.isin(i)]) - clim.sel(month=i))
    return xr.concat(ano_list, dim='time').sortby('time')

def regrid(in_grid,out_grid, var): 
    reg_bil = xe.Regridder(in_grid,out_grid, "bilinear")
    var_bil = reg_bil(in_grid[var])
    return var_bil

def clim(ds):
    return ds.groupby('time.month').mean('time')

def shift_lon(ds, lon_name = 'lon'):
    # Adjust lon values to make sure they are within (-180, 180)
    ds['_longitude_adjusted'] = xr.where(
        ds[lon_name] > 180,
        ds[lon_name] - 360,
        ds[lon_name])

    # reassign the new coords to as the main lon coords
    # and sort DataArray using new coordinate values
    ds = (
        ds
        .swap_dims({lon_name: '_longitude_adjusted'})
        .sel(**{'_longitude_adjusted': sorted(ds._longitude_adjusted)})
        .drop_vars(lon_name))

    ds = ds.rename({'_longitude_adjusted': lon_name})
    return ds

def calculate_nino34_index(ds):
    # ds needs to be tos-anomalies
    # 170° - 120°W -> 190 - 240°E
    nino = ds.sel(lon=slice(190,240), lat=slice(-5,5)).mean(['lon', 'lat']).compute()
    return nino

def calculate_nao(ds):
    ds_south = ds.sel(lat=slice(20,55), lon=slice(-90,60)).mean(['lon','lat'])
    ds_north = ds.sel(lat=slice(55,90), lon=slice(-90,60)).mean(['lon','lat'])
    nao = (ds_south - ds_north).compute()
    nao = nao.zg.to_dataset(name='zg')
    return nao


def calculate_djf_mean(ds):
    ds_seasons = ds.resample(time='QE-FEB').mean()
    ds_djf = ds_seasons.sel(time=(ds_seasons['time.season'] == 'DJF'))
    return ds_djf

In [2]:
outdir = "/sto4/data/Intermediate/VolMIP/"
#models = ['CanESM5', 'MIROC-ES2L', 'MPI-ESM1-2-LR', 'GISS-E2-1-G', 'UKESM1-0-LL']
models = ['GISS-E2-1-G']
# member_id = ['r1i1p2f1', 'r1i1p1f2', 'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f2']
member_id = ['r1i1p1f2']

In [ ]:
for model, id in zip(models, member_id):

    cat=ESGFCatalog()
    metagrid_search = cat.search(activity_id='CMIP', experiment_id='piControl', source_id=model,
           variable_id=['ta', 'ua'], latest=True, table_id=['Amon'], member_id=id, grid_label='gn')
    print(metagrid_search)
    ds = metagrid_search.to_path_dict(outdir + 'piControl/' + model + '/piControl_' + model + '_NAO_djf_'+ start + '-' + end + '.nc')

   Searching indices:   0%|          |0/1 [       ?index/s]

Summary information for 2 results:
mip_era                 [CMIP6]
activity_drs             [CMIP]
institution_id      [NASA-GISS]
source_id         [GISS-E2-1-G]
experiment_id       [piControl]
member_id            [r1i1p1f2]
table_id                 [Amon]
variable_id            [ta, ua]
grid_label                 [gn]
dtype: object


Get file information:   0%|          |0/1 [       ?index/s]

In [6]:
ds

{'ta': [PosixPath('/sto0/user/Kperny/.esgf/CMIP6/CMIP/NASA-GISS/GISS-E2-1-G/piControl/r1i1p1f2/Amon/ta/gn/v20190903/ta_Amon_GISS-E2-1-G_piControl_r1i1p1f2_gn_755001-760012.nc'),
  PosixPath('/sto0/user/Kperny/.esgf/CMIP6/CMIP/NASA-GISS/GISS-E2-1-G/piControl/r1i1p1f2/Amon/ta/gn/v20190903/ta_Amon_GISS-E2-1-G_piControl_r1i1p1f2_gn_760101-765012.nc'),
  PosixPath('/sto0/user/Kperny/.esgf/CMIP6/CMIP/NASA-GISS/GISS-E2-1-G/piControl/r1i1p1f2/Amon/ta/gn/v20190903/ta_Amon_GISS-E2-1-G_piControl_r1i1p1f2_gn_765101-770012.nc'),
  PosixPath('/sto0/user/Kperny/.esgf/CMIP6/CMIP/NASA-GISS/GISS-E2-1-G/piControl/r1i1p1f2/Amon/ta/gn/v20190903/ta_Amon_GISS-E2-1-G_piControl_r1i1p1f2_gn_770101-775012.nc'),
  PosixPath('/sto0/user/Kperny/.esgf/CMIP6/CMIP/NASA-GISS/GISS-E2-1-G/piControl/r1i1p1f2/Amon/ta/gn/v20190903/ta_Amon_GISS-E2-1-G_piControl_r1i1p1f2_gn_775101-780012.nc'),
  PosixPath('/sto0/user/Kperny/.esgf/CMIP6/CMIP/NASA-GISS/GISS-E2-1-G/piControl/r1i1p1f2/Amon/ta/gn/v20190903/ta_Amon_GISS-E2-1-G_piCo

In [ ]:
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
ds_ta = xr.open_mfdataset(ds['ta'], decode_times=time_coder)

In [ ]:
ta_zonal = ds_ta.mean('lon')
nino.to_netcdf(outdir + 'piControl/' + model + '/ta.nc')


<xarray.Dataset> Size: 71MB
Dimensions:    (time: 12000, bnds: 2, lat: 64, plev: 19)
Coordinates:
  * time       (time) object 96kB 5201-01-16 12:00:00 ... 6200-12-16 12:00:00
  * plev       (plev) float64 152B 1e+05 9.25e+04 8.5e+04 ... 1e+03 500.0 100.0
  * lat        (lat) float64 512B -87.86 -85.1 -82.31 ... 82.31 85.1 87.86
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) object 192kB dask.array<chunksize=(1, 2), meta=np.ndarray>
    lat_bnds   (time, lat, bnds) float64 12MB dask.array<chunksize=(2400, 64, 2), meta=np.ndarray>
    lon_bnds   (time, bnds) float64 192kB dask.array<chunksize=(2400, 2), meta=np.ndarray>
    ta         (time, plev, lat) float32 58MB dask.array<chunksize=(1, 19, 64), meta=np.ndarray>

In [ ]:
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
ds_tos = xr.open_mfdataset(ds['Omon.tos'], decode_times=time_coder)

# regrid to regular lat lon grid
ds_tos_reg = regrid(ds_tos, ds_zg, 'tos')

# sst climatology
tos_clim = clim(ds_tos_reg)
tos_clim.to_netcdf(outdir + 'piControl/' + model + '/tos_clim_full.nc')

# calculate nino index
nino = calculate_nino34_index(ds_tos_reg)
nino = nino.to_dataset(name='tos')
nino_djf = calculate_djf_mean(nino)

# build anomalies & calculate rsst nino index
ds_ano = anomaly_calculation(ds_tos_reg, tos_clim)
rsstnino = calculate_nino34_index(ds_ano)
rsstnino = rsstnino.to_dataset(name='tos')
rsstnino_djf = calculate_djf_mean(rsstnino)

nino.to_netcdf(outdir + 'piControl/' + model + '/nino34_full.nc')
nino_djf.to_netcdf(outdir + 'piControl/' + model + '/nino34_djf.nc')

rsstnino.to_netcdf(outdir + 'piControl/' + model + '/rsstnino34_full.nc')
rsstnino_djf.to_netcdf(outdir + 'piControl/' + model + '/rsstnino34_djf.nc')

In [ ]:
str(ds_zg.time[0].values)

'1850-01-16 12:00:00'

In [ ]:
# shift lon to °W-°E so there is no problem at 0° 
ds_prep = shift_lon(ds_zg.sel(plev=50000, lat=slice(0,100)))

# calculate nao index
nao = calculate_nao(ds_prep)
nao_djf = calculate_djf_mean(nao)

nao.to_netcdf(outdir + 'piControl/' + model + '/nao_full.nc')
nao_djf.to_netcdf(outdir + 'piControl/' + model + '/nao_djf.nc')

In [ ]:
from intake_esgf import ESGFCatalog
cat=ESGFCatalog()

metagrid_search=cat.search(activity_id='VolMIP', experiment_id='volc-pinatubo-full', source_id=models[4],
                           variable_id=['ta', 'ua'], table_id='Eday', latest=True)
metagrid_search

   Searching indices:   0%|          |0/1 [       ?index/s]

Summary information for 54 results:
mip_era                                                     [CMIP6]
activity_drs                                               [VolMIP]
institution_id                                               [NERC]
source_id                                             [UKESM1-0-LL]
experiment_id                                  [volc-pinatubo-full]
member_id         [r10i1p1f2, r11i1p1f2, r12i1p1f2, r13i1p1f2, r...
table_id                                                     [Eday]
variable_id                                                [ta, ua]
grid_label                                                     [gn]
dtype: object

In [ ]:
ds = metagrid_search.to_path_dict()

Get file information:   0%|          |0/1 [       ?index/s]

In [ ]:
ds

{'r10i1p1f2.ta': [PosixPath('/home/kperny/.esgf/CMIP6/VolMIP/NERC/UKESM1-0-LL/volc-pinatubo-full/r10i1p1f2/Eday/ta/gn/v20230810/ta_Eday_UKESM1-0-LL_volc-pinatubo-full_r10i1p1f2_gn_19910601-19940630.nc')],
 'r10i1p1f2.ua': [PosixPath('/home/kperny/.esgf/CMIP6/VolMIP/NERC/UKESM1-0-LL/volc-pinatubo-full/r10i1p1f2/Eday/ua/gn/v20230810/ua_Eday_UKESM1-0-LL_volc-pinatubo-full_r10i1p1f2_gn_19910601-19940630.nc')],
 'r11i1p1f2.ta': [PosixPath('/home/kperny/.esgf/CMIP6/VolMIP/NERC/UKESM1-0-LL/volc-pinatubo-full/r11i1p1f2/Eday/ta/gn/v20230810/ta_Eday_UKESM1-0-LL_volc-pinatubo-full_r11i1p1f2_gn_19910601-19940630.nc')],
 'r11i1p1f2.ua': [PosixPath('/home/kperny/.esgf/CMIP6/VolMIP/NERC/UKESM1-0-LL/volc-pinatubo-full/r11i1p1f2/Eday/ua/gn/v20230810/ua_Eday_UKESM1-0-LL_volc-pinatubo-full_r11i1p1f2_gn_19910601-19940630.nc')],
 'r12i1p1f2.ta': [PosixPath('/home/kperny/.esgf/CMIP6/VolMIP/NERC/UKESM1-0-LL/volc-pinatubo-full/r12i1p1f2/Eday/ta/gn/v20231129/ta_Eday_UKESM1-0-LL_volc-pinatubo-full_r12i1p1f2_gn_

In [ ]:
xr.open_dataset('/sto4/data/Intermediate/VolMIP/pinatubo-full/GISS-E2-1-G/ta_Amon_GISS-E2-1-G_volc-pinatubo-full_r1i1p1f1_gn_800001-800212.nc')

FileNotFoundError: [Errno 2] No such file or directory: '/sto4/data/Intermediate/VolMIP/pinatubo-full/GISS-E2-1-G/ta_Amon_GISS-E2-1-G_volc-pinatubo-full_r1i1p1f1_gn_800001-800212.nc'

In [ ]:
import xarray as xr

#ds.values()
ds_ta = xr.open_mfdataset(np.concatenate(list(ds.values())), concat_dim='ens', combine='nested')

In [ ]:
ds_ta

<xarray.Dataset> Size: 23MB
Dimensions:    (ens: 27, time: 37, bnds: 2, lat: 144, plev: 39)
Coordinates:
  * time       (time) object 296B 1991-06-16 00:00:00 ... 1994-06-16 00:00:00
  * plev       (plev) float64 312B 1e+05 9.25e+04 8.5e+04 7e+04 ... 7.0 5.0 3.0
  * lat        (lat) float64 1kB -89.38 -88.12 -86.88 ... 86.88 88.12 89.38
Dimensions without coordinates: ens, bnds
Data variables:
    time_bnds  (ens, time, bnds) object 16kB dask.array<chunksize=(1, 1, 2), meta=np.ndarray>
    lat_bnds   (ens, lat, bnds) float64 62kB dask.array<chunksize=(1, 144, 2), meta=np.ndarray>
    ta         (ens, time, plev, lat) float32 22MB dask.array<chunksize=(1, 1, 39, 144), meta=np.ndarray>
Attributes: (12/46)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            VolMIP
    branch_method:          standard
    branch_time_in_child:   50910.0
    branch_time_in_parent:  462750.0
    creation_date:          2023-08-09T22:43:01Z
    ...                     ...
    variable_id:            ta
    variable_name:          ta
    variant_label:          r10i1p1f2
    license:                CMIP6 model data produced by the Natural Environm...
    cmor_version:           3.7.1
    tracking_id:            hdl:21.14100/8432ce9d-9c52-4fb9-b392-7ff6cbb0f06f

In [ ]:
nao = nao.zg.to_dataset(name='zg')
nao.to_netcdf(outdir + 'piControl/' + model + '/nao_full.nc')

In [ ]:
outdir = "/sto4/data/Intermediate/VolMIP/"
models = ['CanESM5', 'MIROC-ES2L', 'MPI-ESM1-2-LR', 'GISS-E2-1-G', 'UKESM1-0-LL']
# according to member id in VOLMIP!!
member_id = ['r1i1p2f1', 'r1i1p1f2', 'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f2']

for model, id in zip(models, member_id):
    print(model, id)
    # open ESGF catalog
    cat=ESGFCatalog()
    metagrid_search = cat.search(activity_id='CMIP', experiment_id='piControl', source_id=model,
            variable_id=['tos', 'zg'], latest=True, table_id=['Omon','Amon'], member_id=id)
    print(metagrid_search)

CanESM5 r1i1p2f1


   Searching indices:   0%|          |0/1 [       ?index/s]

Summary information for 2 results:
mip_era                [CMIP6]
activity_drs            [CMIP]
institution_id         [CCCma]
source_id            [CanESM5]
experiment_id      [piControl]
member_id           [r1i1p2f1]
table_id          [Amon, Omon]
variable_id          [zg, tos]
grid_label                [gn]
dtype: object
MIROC-ES2L r1i1p1f2


   Searching indices:   0%|          |0/1 [       ?index/s]

Summary information for 3 results:
mip_era                [CMIP6]
activity_drs            [CMIP]
institution_id         [MIROC]
source_id         [MIROC-ES2L]
experiment_id      [piControl]
member_id           [r1i1p1f2]
table_id          [Amon, Omon]
variable_id          [zg, tos]
grid_label           [gn, gr1]
dtype: object
MPI-ESM1-2-LR r1i1p1f1


...

Summary information for 2 results:
mip_era                   [CMIP6]
activity_drs               [CMIP]
institution_id            [MPI-M]
source_id         [MPI-ESM1-2-LR]
experiment_id         [piControl]
member_id              [r1i1p1f1]
table_id             [Amon, Omon]
variable_id             [zg, tos]
grid_label                   [gn]
dtype: object
GISS-E2-1-G r1i1p1f1


   Searching indices:   0%|          |0/1 [       ?index/s]

Summary information for 2 results:
mip_era                 [CMIP6]
activity_drs             [CMIP]
institution_id      [NASA-GISS]
source_id         [GISS-E2-1-G]
experiment_id       [piControl]
member_id            [r1i1p1f1]
table_id           [Amon, Omon]
variable_id           [zg, tos]
grid_label                 [gn]
dtype: object
UKESM1-0-LL r1i1p1f2


   Searching indices:   0%|          |0/1 [       ?index/s]

Summary information for 2 results:
mip_era                 [CMIP6]
activity_drs             [CMIP]
institution_id           [MOHC]
source_id         [UKESM1-0-LL]
experiment_id       [piControl]
member_id            [r1i1p1f2]
table_id           [Amon, Omon]
variable_id           [zg, tos]
grid_label                 [gn]
dtype: object


In [ ]:
xr.open_dataset(outdir + 'piControl/' + model + '/nino34_djf.nc')

<xarray.Dataset> Size: 12kB
Dimensions:                        (time: 1001)
Coordinates:
  * time                           (time) object 8kB 1850-02-28 00:00:00 ... ...
Data variables:
    __xarray_dataarray_variable__  (time) float32 4kB ...